In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen

In [3]:
master = pd.read_csv("../data/processed/quarterly_master.csv", parse_dates=["date"])

log_vars = ["starts_private", "hpi_index", "transactions_sa", "housing_stock_eng"]

adf_df = master[["date"] + log_vars + ["base_rate"]].dropna().copy()
for col in log_vars:
    adf_df[f"log_{col}"] = np.log(adf_df[col])

vecm_vars = ["log_starts_private", "log_hpi_index", "log_transactions_sa",
             "log_housing_stock_eng", "base_rate"]

vecm_df = adf_df[["date"] + [f"log_{c}" if c != "base_rate" else c
                              for c in ["starts_private", "hpi_index",
                                        "transactions_sa", "housing_stock_eng", "base_rate"]]].dropna()
vecm_df.columns = ["date"] + vecm_vars
vecm_df = vecm_df.set_index("date")

print(f"vecm_df: {vecm_df.shape[0]} obs, {vecm_df.shape[1]} variables")
print(f"Sample: {vecm_df.index.min().date()} to {vecm_df.index.max().date()}")

vecm_df: 149 obs, 5 variables
Sample: 1987-01-01 to 2024-01-01


In [4]:
vecm_model = VECM(
    vecm_df,
    k_ar_diff=4,
    coint_rank=1,
    deterministic="ci"
)

vecm_fit = vecm_model.fit()
print(vecm_fit.summary())

Det. terms outside the coint. relation & lagged endog. parameters for equation log_starts_private
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
L1.log_starts_private       -0.4592      0.099     -4.635      0.000      -0.653      -0.265
L1.log_hpi_index             1.3657      0.989      1.381      0.167      -0.573       3.304
L1.log_transactions_sa       0.3196      0.153      2.084      0.037       0.019       0.620
L1.log_housing_stock_eng   229.5031    327.205      0.701      0.483    -411.806     870.813
L1.base_rate                -0.0089      0.033     -0.269      0.788      -0.074       0.056
L2.log_starts_private       -0.4603      0.110     -4.203      0.000      -0.675      -0.246
L2.log_hpi_index            -2.0827      1.301     -1.600      0.110      -4.633       0.468
L2.log_transactions_sa       0.4286      0.157      2.728      0.

/home/adrym/projects/dissertation_housing_supply/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)


In [6]:
from statsmodels.tsa.vector_ar.vecm import VECM

# Re-estimate on post-1995 sample to check stability
vecm_df_post95 = vecm_df["1995-01-01":]

vecm_post95 = VECM(
    vecm_df_post95,
    k_ar_diff=4,
    coint_rank=1,
    deterministic="ci"
).fit()

# Print just the cointegrating vector and alpha
print("POST-1995 SAMPLE")
print("\nCointegrating vector:")
for i, var in enumerate(vecm_vars):
    print(f"  {var}: {vecm_post95.beta[i,0]:.4f}")
print("\nAlpha (adjustment speeds):")
for i, var in enumerate(vecm_vars):
    print(f"  {var}: {vecm_post95.alpha[i,0]:.4f}")

print("\n")
print(vecm_post95.summary())

POST-1995 SAMPLE

Cointegrating vector:
  log_starts_private: 1.0000
  log_hpi_index: 0.2969
  log_transactions_sa: 0.0997
  log_housing_stock_eng: -1.2702
  base_rate: -0.0326

Alpha (adjustment speeds):
  log_starts_private: -0.6626
  log_hpi_index: -0.0332
  log_transactions_sa: -0.1944
  log_housing_stock_eng: 0.0001
  base_rate: -0.0231


Det. terms outside the coint. relation & lagged endog. parameters for equation log_starts_private
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
L1.log_starts_private       -0.0696      0.158     -0.440      0.660      -0.380       0.240
L1.log_hpi_index             2.3662      1.245      1.901      0.057      -0.074       4.806
L1.log_transactions_sa       0.1705      0.157      1.084      0.278      -0.138       0.479
L1.log_housing_stock_eng  1628.3371    568.877      2.862      0.004     513.359    2743.31

/home/adrym/projects/dissertation_housing_supply/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)


In [7]:
from statsmodels.tsa.stattools import zivot_andrews

# Run ZA test on log starts — model='both' allows break in intercept and trend
za_results = {}
for col in ["log_starts_private", "log_hpi_index", "log_transactions_sa",
            "log_housing_stock_eng", "base_rate"]:

    series = adf_df[f"log_{col}"] if col != "base_rate" else adf_df["base_rate"]
    series = series.dropna()

    za_stat, pval, cvs, baselag, breakpoint = zivot_andrews(
        series,
        trim=0.15,        # exclude outer 15% of sample from break search
        regression="both" # allow break in intercept and trend
    )

    break_date = series.index[breakpoint]

    za_results[col] = {
        "ZA Stat": round(za_stat, 3),
        "p-value": round(pval, 3),
        "Break point (obs)": breakpoint,
        "Break date": break_date,
        "1%": round(cvs["1%"], 3),
        "5%": round(cvs["5%"], 3),
        "10%": round(cvs["10%"], 3),
        "Reject H0 (5%)": "Yes" if za_stat < cvs["5%"] else "No"
    }

za_df = pd.DataFrame(za_results).T
print("Zivot-Andrews Structural Break Tests\n")
print(za_df[["ZA Stat", "p-value", "Break date", "5%", "Reject H0 (5%)"]].to_string())

KeyError: 'log_log_starts_private'

# Adding Brexit Dummy

In [8]:
# Brexit dummy: 0 before 2016Q3, 1 from 2016Q3 onwards
vecm_df_post95 = vecm_df["1995-01-01":].copy()
vecm_df_post95["brexit"] = (vecm_df_post95.index >= "2016-07-01").astype(int)

# Separate out the exogenous dummy
endog = vecm_df_post95[vecm_vars]
exog  = vecm_df_post95[["brexit"]]

vecm_brexit = VECM(
    endog,
    exog=exog,
    k_ar_diff=4,
    coint_rank=1,
    deterministic="ci"
).fit()

print(vecm_brexit.summary())

Det. terms outside the coint. relation & lagged endog. parameters for equation log_starts_private
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
exog1                        0.1355      0.047      2.891      0.004       0.044       0.227
L1.log_starts_private       -0.0009      0.183     -0.005      0.996      -0.359       0.357
L1.log_hpi_index             1.6103      1.258      1.280      0.201      -0.856       4.077
L1.log_transactions_sa       0.0942      0.157      0.599      0.549      -0.214       0.403
L1.log_housing_stock_eng  1517.9955    564.078      2.691      0.007     412.422    2623.569
L1.base_rate                 0.0869      0.057      1.513      0.130      -0.026       0.199
L2.log_starts_private       -0.0919      0.168     -0.546      0.585      -0.422       0.238
L2.log_hpi_index            -1.6098      1.666     -0.966      0.

/home/adrym/projects/dissertation_housing_supply/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)


In [9]:
print("COINTEGRATING VECTOR (with Brexit dummy)\n")
for i, var in enumerate(vecm_vars):
    print(f"  {var:<30} {vecm_brexit.beta[i,0]:>8.4f}")

print("\nADJUSTMENT COEFFICIENTS (alpha)\n")
for i, var in enumerate(vecm_vars):
    print(f"  {var:<30} {vecm_brexit.alpha[i,0]:>8.4f}")

print("\nBREXIT DUMMY COEFFICIENTS (effect on each equation)\n")
for i, var in enumerate(vecm_vars):
    print(f"  {var:<30} {vecm_brexit.det_coef[i,0]:>8.4f}")

COINTEGRATING VECTOR (with Brexit dummy)

  log_starts_private               1.0000
  log_hpi_index                    0.1235
  log_transactions_sa             -0.1264
  log_housing_stock_eng            0.1187
  base_rate                       -0.0338

ADJUSTMENT COEFFICIENTS (alpha)

  log_starts_private              -0.7386
  log_hpi_index                   -0.0477
  log_transactions_sa             -0.1670
  log_housing_stock_eng            0.0002
  base_rate                        0.1566

BREXIT DUMMY COEFFICIENTS (effect on each equation)

  log_starts_private               0.1355
  log_hpi_index                    0.0086
  log_transactions_sa              0.0565
  log_housing_stock_eng           -0.0000
  base_rate                        0.1130


Bad results. Need to use post-1995 data